# PKG — Counterparty Locatability
### Can we infer where an external account is, from who it pays?
**PNC Treasury Management · Data Science**

---

## The question this notebook actually answers

The goal is to locate **counterparty** nodes, for which no address exists. We
cannot measure error on them — there is no ground truth. So we run the whole
problem on **customers**, where the address is known, by hiding it and
predicting it back from their neighbours.

The framing matters. At degree 1 the achievable error is not a number, it is
**bimodal**: a single neighbour that is a neighbourhood restaurant pins the
target within a few km; a single neighbour that is a payment processor gives
you the population prior and nothing else. No estimator repairs the second
case.

So the deliverable is **not a centroid for every node**. It is:

1. an **error-vs-evidence curve** — how error falls as located neighbours
   accumulate, measured from k=1 upward;
2. a **locatability gate** — which single-neighbour cases are locatable at
   all, and at what radius;
3. a **neighbour informativeness ranking** — which sectors, sizes and hub
   classes carry location, and which destroy it.

A point estimate emitted for every node, with no way to tell the two regimes
apart, is worse than useless downstream. Confidence is the primary output.

## Design decisions, stated up front

| Decision | Choice | Why |
|---|---|---|
| Kernel centre | neighbour's **counterparty centroid**, not its registered pin | the target *is* one of the neighbour's counterparties; the relevant distribution is the cloud, not the pin |
| Kernel width | neighbour's own angular dispersion | a hub gets a near-flat kernel and self-mutes — no exclusion list needed |
| Leakage | **exact leave-one-out** on both centre and width | the target is inside the neighbour's cloud statistic; at low neighbour degree this is circular and dominates |
| Weighting | uniform / amount / inverse-variance, compared head to head | Block F weights by dollars to describe a footprint; dollars are close to *anti*-correlated with locating power |
| Ground truth | registered pin, **and** a restricted set where the pin is representative | a wide-footprint firm's registered pin is itself noisy truth |
| Baseline to beat | the **PNC deposit-footprint prior** | much apparent skill is just "PNC customers are in PA/OH" |

## Switches this notebook is built around

- `WINDOW` — `last3` vs `all`. Registered location is **current-state applied
  to all history**, so pooling 23 months adds edges (good) and stales the
  attribution for anyone who moved (bad). The difference between the two runs
  is the **net** of those two effects, which is the honest thing to measure.
- `VERSION` — `V0` (hubs in) vs `P99_9` (hubs out). Run V0 first: some hubs are
  government accounts tightly bound to the geography they serve, and the
  question is not whether to exclude hubs but **which hubs locate**.

Run one configuration at a time; each writes to its own folder and §10
compares them.

---
## 0. Setup

In [ ]:
import os, re, math, time, json, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
import plotly.io as pio
pio.renderers.default = "notebook"; pio.templates.default = "plotly_white"
PALETTE = px.colors.qualitative.Safe

# ================================ CONFIG ==================================
WINDOW  = "last3"       # "last3" | "all"   <- staleness vs volume experiment
VERSION = "V0"          # "V0"    | "P99_9" <- hubs in / hubs out

METRIC_TABLE  = "bdahd01p_dlcdi1_cdi_tm.cust_c2c_metrics"
METRIC_SOURCE = "table"                 # "table" | "parquet"
METRIC_GLOB   = "/user/pk36814/metrics/node/*.parquet"
EDGE_GLOB     = "../data/cust_*.csv"    # snapshots: source,dest,amount,volume
EDGE_FORMAT   = "csv"                   # "csv" | "parquet"

OUT_ROOT = "../metrics/locatability"
OUT = os.path.join(OUT_ROOT, f"{VERSION}_{WINDOW}")
os.makedirs(OUT, exist_ok=True)

# analysis parameters
K_MAX          = 20      # largest neighbour count evaluated in the curve
HARNESS_REPS   = 5       # random permutations per target (see §5 cost note)
HIT_RADII_KM   = [25, 50, 250]
MIN_R_BAR      = 1e-9    # degenerate resultant guard
REPRESENTATIVE_GAP_KM = 25   # "pin is meaningful" cut for the clean truth set
SEED = 17

print(f"VERSION={VERSION}  WINDOW={WINDOW}\nout -> {OUT}")

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window as W
from pyspark.sql.types import StringType, DoubleType

spark = (SparkSession.builder.appName(f"pkg_locatability_{VERSION}_{WINDOW}")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.shuffle.partitions", "600")
         .config("spark.sql.autoBroadcastJoinThreshold", str(64 * 1024 * 1024))
         .enableHiveSupport().getOrCreate())
spark.sparkContext.setLogLevel("WARN")
R_EARTH_KM = 6371.0088

def to_pd(sdf, label="", max_rows=3_000_000):
    t = time.time(); n = sdf.count()
    if n > max_rows:
        raise MemoryError(f"[{label}] {n:,} rows > max_rows={max_rows:,}; "
                          f"aggregate in Spark first.")
    df = sdf.toPandas()
    print(f"[{label}] {n:,} rows | {time.time()-t:,.1f}s")
    return df

def unit_cols(df, lat="lat", lon="lon", pre=""):
    "lat/lon degrees -> 3D unit vector. Spherical means done properly."
    la, lo = F.radians(F.col(lat)), F.radians(F.col(lon))
    return (df.withColumn(pre+"ux", F.cos(la) * F.cos(lo))
              .withColumn(pre+"uy", F.cos(la) * F.sin(lo))
              .withColumn(pre+"uz", F.sin(la)))

def gc_km(lat1, lon1, lat2, lon2):
    "Great-circle distance as a Spark column expression."
    p1, p2 = F.radians(lat1), F.radians(lat2)
    dp, dl = p2 - p1, F.radians(lon2) - F.radians(lon1)
    a = (F.sin(dp/2)**2 + F.cos(p1)*F.cos(p2)*F.sin(dl/2)**2)
    return F.lit(2*R_EARTH_KM) * F.asin(F.sqrt(F.least(a, F.lit(1.0))))

def vec_to_latlon(df, vx, vy, vz, out_lat, out_lon):
    n = F.sqrt(F.col(vx)**2 + F.col(vy)**2 + F.col(vz)**2)
    return (df.withColumn("_n", n)
              .withColumn(out_lat, F.degrees(F.asin(F.col(vz)/F.col("_n"))))
              .withColumn(out_lon, F.degrees(F.atan2(F.col(vy), F.col(vx))))
              .drop("_n"))

### 0.1 The bandwidth estimator, and why it leave-one-out subtracts cleanly

Each neighbour *j* contributes a kernel. Its width should be *j*'s own
counterparty dispersion — wide for a processor, tight for a corner shop.

Using **mean resultant length** makes the leave-one-out exact and cheap. For
counterparties with unit vectors `u` and weights `w`:

```
S  = Σ w                 V  = Σ w·u              R̄ = |V| / S
```

Removing target *i* is a subtraction, not a recomputation:

```
S' = S − w_i            V' = V − w_i·u_i         R̄' = |V'| / S'
```

- **LOO centre** = `V' / |V'|` projected back to the sphere
- **LOO width** = `R_EARTH · sqrt(2(1 − R̄'))` — the standard angular-dispersion
  approximation, in km

This is the same quantity as Block F's `geo_R`, so the bandwidth is
consistent with the published metric rather than a parallel invention. It
also means **a neighbour whose only located counterparty is the target
collapses to S'=0 and drops out automatically** — which is correct, it has no
independent information.

---
## 1. Metrics dimension and the hub set

In [ ]:
SDF_M = (spark.table(METRIC_TABLE) if METRIC_SOURCE == "table"
         else spark.read.parquet(METRIC_GLOB))
COLS = [c.lower() for c in SDF_M.columns]
SDF_M = SDF_M.toDF(*COLS)

def have(*c):
    for x in c:
        if x in COLS: return x
    return None

# The join key. The metric table carries mdm_id; snapshots carry source/dest.
NODE = have("mdm_id", "node")
TIME, VER = have("time_key"), have("version")
assert NODE and TIME and VER, f"missing keys: {NODE}, {TIME}, {VER}"
print(f"join key = {NODE}  |  dtype = {dict(SDF_M.dtypes)[NODE]}")

MONTHS = sorted(r[0] for r in SDF_M.select(TIME).distinct().collect())
WIN_MONTHS = MONTHS[-3:] if WINDOW == "last3" else MONTHS
REF_MONTH = MONTHS[-1]
print(f"{len(MONTHS)} months available; window '{WINDOW}' uses "
      f"{len(WIN_MONTHS)}: {WIN_MONTHS[0]} .. {WIN_MONTHS[-1]}")

In [ ]:
# ---- hub set = present in V0 but ablated out of P99_9, any month in window --
v0  = (SDF_M.filter((F.col(VER) == "V0") & F.col(TIME).isin(WIN_MONTHS))
            .select(F.col(NODE).cast("string").alias("node")).distinct())
p99 = (SDF_M.filter((F.col(VER) == "P99_9") & F.col(TIME).isin(WIN_MONTHS))
            .select(F.col(NODE).cast("string").alias("node")).distinct())
HUBS = v0.join(p99, "node", "left_anti").cache()
n_hub = HUBS.count()
print(f"hub nodes (in V0, ablated at P99_9): {n_hub:,} of {v0.count():,}")

# ---- node attributes: latest month in the window, V0 rows (superset) -------
attr_cols = [c for c in [NODE, "lat", "lon", "geo_status", "state", "zip3",
                         "naics2", "naics_desc", "cust_name", "node_type",
                         "entity_type", "party_type", "in_degree", "out_degree",
                         "in_strength", "out_strength", "geo_spread_km",
                         "geo_reach_p50_km", "geo_registered_vs_flow_km",
                         "geo_r", "geo_cov_amt_in", "geo_cov_amt_out"]
             if c in COLS]
ATTR = (SDF_M.filter((F.col(VER) == "V0") & (F.col(TIME) == WIN_MONTHS[-1]))
             .select(*attr_cols)
             .withColumn("node", F.col(NODE).cast("string")))
if NODE != "node": ATTR = ATTR.drop(NODE)
ATTR = (ATTR.withColumn("is_hub", F.col("node").isin(
            [r[0] for r in HUBS.limit(0).collect()]) if False else F.lit(0))
            .join(HUBS.withColumn("is_hub_j", F.lit(1)), "node", "left")
            .withColumn("is_hub", F.coalesce(F.col("is_hub_j"), F.lit(0)))
            .drop("is_hub_j"))
ATTR = ATTR.withColumn("has_geo",
                       (F.col("geo_status") == "valid").cast("int")).cache()
print(f"attribute rows: {ATTR.count():,} | located: "
      f"{ATTR.filter('has_geo = 1').count():,}")
ATTR.select("node","lat","lon","geo_status","naics2","node_type","is_hub").show(4)

---
## 2. Edges, pooled over the window

In [ ]:
if EDGE_FORMAT == "csv":
    E = (spark.read.option("header", True).csv(EDGE_GLOB)
              .withColumn("amount", F.col("amount").cast("double"))
              .withColumn("volume", F.col("volume").cast("double")))
else:
    E = spark.read.parquet(EDGE_GLOB)
E = E.toDF(*[c.lower() for c in E.columns])
# month comes from the filename when the snapshot has no time column
if "time_key" not in E.columns:
    E = E.withColumn("time_key",
                     F.regexp_extract(F.input_file_name(), r"(\d{4}-\d{2})", 1))
E = (E.withColumn("source", F.col("source").cast("string"))
       .withColumn("dest",   F.col("dest").cast("string"))
       .filter(F.col("time_key").isin(WIN_MONTHS))
       .filter(F.col("source") != F.col("dest")))
print(f"edge rows in window: {E.count():,}")

# ---- JOIN INTEGRITY. The 2026-07 incident was an int64/str mismatch that
# typed every counterparty 'unknown' while producing plausible output. Fail
# fast rather than discover it in a chart.
probe = E.select("source").limit(200_000).distinct()
mrate = (probe.join(ATTR.select("node"), probe.source == F.col("node"), "left")
              .agg(F.avg(F.col("node").isNotNull().cast("double"))).first()[0])
print(f"edge->metric join match rate: {mrate:.2%}")
assert mrate > 0.5, ("join match rate below 50% — check that source/dest and "
                     f"{NODE} are both strings and share an id space")

In [ ]:
# ---- undirected pair table, pooled across the window -----------------------
# Direction is kept as separate amount columns: in- and out-locality differ
# (home_state_share_in vs _out are not the same), so the harness must be able
# to ask which side locates better.
fwd = E.select(F.col("source").alias("a"), F.col("dest").alias("b"),
               F.col("amount").alias("amt_ab"), F.lit(0.0).alias("amt_ba"),
               "time_key")
rev = E.select(F.col("dest").alias("a"), F.col("source").alias("b"),
               F.lit(0.0).alias("amt_ab"), F.col("amount").alias("amt_ba"),
               "time_key")
PAIR = (fwd.union(rev).groupBy("a", "b")
          .agg(F.sum("amt_ab").alias("amt_out"),
               F.sum("amt_ba").alias("amt_in"),
               F.countDistinct("time_key").alias("n_months")))
PAIR = PAIR.withColumn("amt", F.col("amt_out") + F.col("amt_in"))

if VERSION == "P99_9":
    # ablate hub-touching edges rather than filtering nodes: the rung is a
    # property of the graph, not of the dimension
    PAIR = (PAIR.join(HUBS.withColumn("hb", F.lit(1)),
                      PAIR.b == HUBS.node, "left").filter("hb is null")
                .drop("node", "hb")
                .join(HUBS.withColumn("ha", F.lit(1)),
                      F.col("a") == F.col("node"), "left").filter("ha is null")
                .drop("node", "ha"))
PAIR = PAIR.cache()
print(f"directed-pair rows (a -> its neighbour b): {PAIR.count():,}")

## 3. Neighbour cloud aggregates, and the exact leave-one-out

`NB` holds, for every node *b* that could serve as evidence, the totals over
**its located counterparties**. `PAIR_LOO` then subtracts the target's own
contribution, giving each (target *a*, neighbour *b*) pair a centre and width
that were computed **without seeing the answer**.

Both a count-weighted and an amount-weighted cloud are carried, because the
choice is an open question: Block F weights by dollars to describe a
customer's economic footprint, but for *locating* an unknown node the largest
edge is often the least informative one.

In [ ]:
LOC = ATTR.filter("has_geo = 1").select("node", "lat", "lon")
LOC = unit_cols(LOC).select("node", "lat", "lon", "ux", "uy", "uz")

# every pair where the NEIGHBOUR side (b) sees a located counterparty (a)
CP = (PAIR.join(LOC.withColumnRenamed("node", "a")
                   .withColumnRenamed("lat", "a_lat")
                   .withColumnRenamed("lon", "a_lon"), "a"))

NB = (CP.groupBy("b").agg(
        F.count("*").alias("n_cp_loc"),
        F.sum("amt").alias("amt_loc"),
        F.sum("ux").alias("cV_x"), F.sum("uy").alias("cV_y"),
        F.sum("uz").alias("cV_z"),
        F.sum(F.col("ux")*F.col("amt")).alias("aV_x"),
        F.sum(F.col("uy")*F.col("amt")).alias("aV_y"),
        F.sum(F.col("uz")*F.col("amt")).alias("aV_z"))).cache()
print(f"nodes usable as evidence (>=1 located counterparty): {NB.count():,}")

In [ ]:
def loo_block(df, pre, S, Vx, Vy, Vz, w):
    """Subtract the target's own contribution, then rebuild centre and width.

    Exact leave-one-out. Without this the neighbour's cloud statistic contains
    the very node being predicted; at low neighbour degree that is circular and
    it is the single largest source of optimism in this kind of analysis.
    A neighbour whose only located counterparty IS the target collapses to
    S'=0 and drops out, which is the correct behaviour.
    """
    d = (df.withColumn(f"{pre}S",  F.col(S)  - F.col(w))
           .withColumn(f"{pre}Vx", F.col(Vx) - F.col(w)*F.col("ux"))
           .withColumn(f"{pre}Vy", F.col(Vy) - F.col(w)*F.col("uy"))
           .withColumn(f"{pre}Vz", F.col(Vz) - F.col(w)*F.col("uz")))
    d = d.withColumn(f"{pre}Vn", F.sqrt(F.col(f"{pre}Vx")**2 +
                                        F.col(f"{pre}Vy")**2 +
                                        F.col(f"{pre}Vz")**2))
    d = (d.withColumn(f"{pre}rbar", F.when(F.col(f"{pre}S") > 0,
                        F.least(F.lit(1.0), F.col(f"{pre}Vn")/F.col(f"{pre}S"))))
           .withColumn(f"{pre}lat", F.when(F.col(f"{pre}Vn") > MIN_R_BAR,
                        F.degrees(F.asin(F.col(f"{pre}Vz")/F.col(f"{pre}Vn")))))
           .withColumn(f"{pre}lon", F.when(F.col(f"{pre}Vn") > MIN_R_BAR,
                        F.degrees(F.atan2(F.col(f"{pre}Vy"), F.col(f"{pre}Vx"))))))
    return d.withColumn(f"{pre}bw_km", F.lit(R_EARTH_KM) * F.sqrt(
        F.greatest(F.lit(0.0), F.lit(2.0)*(F.lit(1.0) - F.col(f"{pre}rbar")))))

# kc_ = count-weighted kernel, ka_ = amount-weighted kernel.
# a_lat / a_lon stay reserved for the TARGET's true position.
L = (CP.join(NB, "b").withColumn("w1", F.lit(1.0))
       .transform(lambda d: loo_block(d, "kc_", "n_cp_loc", "cV_x", "cV_y",
                                      "cV_z", "w1"))
       .transform(lambda d: loo_block(d, "ka_", "amt_loc", "aV_x", "aV_y",
                                      "aV_z", "amt")))
L = (L.withColumn("err_cnt_km", gc_km(F.col("a_lat"), F.col("a_lon"),
                                      F.col("kc_lat"), F.col("kc_lon")))
       .withColumn("err_amt_km", gc_km(F.col("a_lat"), F.col("a_lon"),
                                       F.col("ka_lat"), F.col("ka_lon")))
       .filter(F.col("kc_S") > 0))

# neighbour attributes for the informativeness analysis (§7)
NBATTR = (ATTR.select(F.col("node").alias("b"),
                      F.col("naics2").alias("b_naics2"),
                      F.col("node_type").alias("b_node_type"),
                      F.col("state").alias("b_state"),
                      F.col("cust_name").alias("b_name"),
                      F.col("is_hub").alias("b_is_hub"),
                      (F.coalesce(F.col("in_degree"), F.lit(0)) +
                       F.coalesce(F.col("out_degree"), F.lit(0))).alias("b_deg")))
L = L.join(F.broadcast(NBATTR) if False else NBATTR, "b", "left").cache()
print(f"LOO pair rows (one degree-1 experiment each): {L.count():,}")
L.select("a","b","a_lat","a_lon","kc_lat","kc_lon","kc_bw_km","kc_S",
         "err_cnt_km","err_amt_km","b_naics2","b_is_hub").show(5)

---
## 4. Baselines

Nothing below means anything without these. The one that matters is the
**footprint prior**: PNC's customers are concentrated in a handful of states,
so a model that has learned only that will look skilful. Any estimator that
does not beat this decisively has produced nothing.

In [ ]:
prior = (LOC.agg(F.avg("ux").alias("x"), F.avg("uy").alias("y"),
                 F.avg("uz").alias("z")).first())
pn = math.sqrt(prior.x**2 + prior.y**2 + prior.z**2)
PRIOR_LAT = math.degrees(math.asin(prior.z/pn))
PRIOR_LON = math.degrees(math.atan2(prior.y, prior.x))
print(f"PNC footprint prior centroid: {PRIOR_LAT:.3f}, {PRIOR_LON:.3f}")

base = (LOC.withColumn("err_km", gc_km(F.col("lat"), F.col("lon"),
                                       F.lit(PRIOR_LAT), F.lit(PRIOR_LON)))
           .agg(F.expr("percentile_approx(err_km, 0.5)").alias("median_km"),
                F.expr("percentile_approx(err_km, 0.9)").alias("p90_km"),
                *[F.avg((F.col("err_km") <= r).cast("double")).alias(f"hit_{r}")
                  for r in HIT_RADII_KM]).first().asDict())
BASELINE = {"estimator": "footprint_prior", **base}
print(json.dumps(BASELINE, indent=2, default=float))
print("\nEvery number later in this notebook is only interesting relative to "
      "these. A median of a few hundred km here is the bar to clear.")

---
## 5. Degree 1 — the population that matters

Every located (customer, customer) pair **is** a degree-1 experiment: hide the
target, predict it from that one neighbour alone, measure the error. There are
millions of them, so no subsampling and no Monte Carlo is needed for this
section — it is a census.

This is the regime the counterparty population actually lives in.

In [ ]:
d1 = (L.agg(F.count("*").alias("n"),
            *[x for r in HIT_RADII_KM for x in
              (F.avg((F.col("err_cnt_km") <= r).cast("double")).alias(f"cnt_hit_{r}"),
               F.avg((F.col("err_amt_km") <= r).cast("double")).alias(f"amt_hit_{r}"))],
            F.expr("percentile_approx(err_cnt_km, 0.5)").alias("cnt_median"),
            F.expr("percentile_approx(err_cnt_km, 0.9)").alias("cnt_p90"),
            F.expr("percentile_approx(err_amt_km, 0.5)").alias("amt_median"),
            F.expr("percentile_approx(err_amt_km, 0.9)").alias("amt_p90"))
        .first().asDict())
print(json.dumps({k: (float(v) if v is not None else None)
                  for k, v in d1.items()}, indent=2))
print(f"\nfootprint-prior median for comparison: {BASELINE['median_km']:,.0f} km")
print("\nIf the amount-weighted kernel is worse than the count-weighted one, "
      "that is the answer to whether dollars carry location: they do not. The "
      "largest edge is disproportionately a processor.")

In [ ]:
# The distribution, not the average — this is the bimodality claim, tested.
h = (L.select(F.when(F.col("err_cnt_km") < 1, 0.0)
               .otherwise(F.log10(F.col("err_cnt_km"))).alias("le"))
       .withColumn("bin", F.floor(F.col("le")*4)/4)
       .groupBy("bin").count().orderBy("bin"))
hp = to_pd(h, "error histogram")
hp["km"] = 10**hp["bin"]
fig = px.bar(hp, x="bin", y="count",
             title="Degree-1 error distribution (log10 km) — is it bimodal?",
             color_discrete_sequence=PALETTE)
for r in HIT_RADII_KM:
    fig.add_vline(x=math.log10(r), line_dash="dash", line_color="crimson",
                  annotation_text=f"{r} km")
fig.update_layout(height=420, xaxis_title="log10 error km", yaxis_title="pairs")
fig.show()
print("Two modes would mean the population splits into locatable and "
      "unlocatable rather than sitting on a continuum — which is exactly what "
      "a gate is for, and what a single accuracy number would hide.")

---
## 6. What makes a neighbour informative

The claim to test: **kernel width predicts error, and everything else acts
through it.** Sector, size and hub status should matter only to the extent
they change how spread out the neighbour's counterparties are.

If that holds, the gate needs one feature and is trivially explainable. If it
does not, the residual structure is where the modelling work is.

In [ ]:
def qbucket(df, col, n_bins=10, rel_err=0.01):
    """Decile bucketing via approxQuantile, NOT ntile.

    `ntile` over an unpartitioned window sorts the entire frame into a single
    partition — Spark warns about it, and on a pair table with tens of
    millions of rows it is a driver-side collapse rather than a slowdown.
    approxQuantile is one scan, and the bucket assignment is a cheap map.
    """
    qs = sorted(set(df.approxQuantile(col, [i/n_bins for i in range(1, n_bins)],
                                      rel_err)))
    e = F.when(F.col(col).isNull(), F.lit(None).cast("int"))
    for i, v in enumerate(qs):
        e = e.when(F.col(col) <= F.lit(v), F.lit(i))
    return df.withColumn("bin", e.otherwise(F.lit(len(qs)))), qs

def by_bin(col, label, n_bins=10, categorical=False, min_n=500):
    d = L.filter(F.col("err_cnt_km").isNotNull())
    if categorical:
        d = d.withColumn("bin", F.col(col).cast("string"))
    else:
        d, _ = qbucket(d, col, n_bins)
    g = (d.groupBy("bin").agg(
            F.count("*").alias("n"),
            F.avg(col).alias("bin_value") if not categorical
                else F.lit(None).cast("double").alias("bin_value"),
            F.expr("percentile_approx(err_cnt_km, 0.5)").alias("median_err_km"),
            F.expr("percentile_approx(err_cnt_km, 0.9)").alias("p90_err_km"),
            *[F.avg((F.col("err_cnt_km") <= r).cast("double")).alias(f"hit_{r}")
              for r in HIT_RADII_KM])
          .filter(F.col("n") >= min_n).orderBy("bin"))
    out = to_pd(g, label); out["feature"] = label
    return out

BW = by_bin("kc_bw_km", "kernel bandwidth decile")
print(BW.to_string(index=False))
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=BW.bin.astype(str), y=BW.n, name="pairs", marker_color="#dfe6e9")
fig.add_scatter(x=BW.bin.astype(str), y=BW.median_err_km, name="median err km",
                mode="lines+markers", line=dict(width=3), secondary_y=True)
fig.add_scatter(x=BW.bin.astype(str), y=BW[f"hit_{HIT_RADII_KM[1]}"]*1000,
                name=f"hit@{HIT_RADII_KM[1]}km (x1000)", mode="lines+markers",
                line=dict(dash="dot"), secondary_y=True)
fig.update_layout(height=430, title="Neighbour kernel bandwidth vs prediction "
                  "error — the core relationship", xaxis_title="bandwidth decile")
fig.show()

In [ ]:
DEG  = by_bin("b_deg", "neighbour degree decile")
AMT  = by_bin("amt", "edge amount decile")
MON  = by_bin("n_months", "months active", categorical=True)
HUB  = by_bin("b_is_hub", "hub flag", categorical=True)
NAI  = by_bin("b_naics2", "neighbour NAICS2", categorical=True, min_n=2000)

fig = make_subplots(rows=2, cols=2, subplot_titles=(
    "neighbour degree decile", "edge amount decile",
    "months the relationship was active", "hub flag"))
for i, (d, r, c) in enumerate([(DEG,1,1), (AMT,1,2), (MON,2,1), (HUB,2,2)]):
    fig.add_scatter(x=d.bin.astype(str), y=d.median_err_km, mode="lines+markers",
                    name=d.feature.iloc[0], row=r, col=c, showlegend=False,
                    line=dict(color=PALETTE[i], width=3))
fig.update_layout(height=620, title="Median degree-1 error by neighbour "
                  "attribute (each panel: lower is more informative)")
fig.show()

NAI = NAI.sort_values("median_err_km")
fig = px.bar(NAI, x="bin", y="median_err_km", hover_data=["n", "hit_50"],
             title="Which sectors locate? Median degree-1 error by neighbour "
                   "NAICS2 (ordered)", color_discrete_sequence=PALETTE)
fig.update_layout(height=430, xaxis_title="neighbour naics2",
                  yaxis_title="median error km")
fig.show()
print(NAI[["bin","n","median_err_km","p90_err_km","hit_50"]].to_string(index=False))

In [ ]:
# Does sector act ONLY through bandwidth? Compare raw sector effect against
# the sector effect after conditioning on the bandwidth decile.
d, _ = qbucket(L.filter(F.col("err_cnt_km").isNotNull()), "kc_bw_km", 10)
d = d.withColumnRenamed("bin", "bwd")
raw = (d.groupBy("b_naics2").agg(F.expr("percentile_approx(err_cnt_km,0.5)")
        .alias("raw_median"), F.count("*").alias("n")).filter("n >= 2000"))
cond = (d.groupBy("b_naics2", "bwd")
          .agg(F.expr("percentile_approx(err_cnt_km,0.5)").alias("m"),
               F.count("*").alias("n")).filter("n >= 200")
          .groupBy("b_naics2").agg(
              (F.sum(F.col("m")*F.col("n"))/F.sum("n")).alias("cond_median")))
S = to_pd(raw.join(cond, "b_naics2"), "sector conditional")
S["shrinkage"] = 1 - (S.cond_median.std() / max(S.raw_median.std(), 1e-9))
fig = px.scatter(S, x="raw_median", y="cond_median", text="b_naics2", size="n",
                 title="Sector effect before vs after conditioning on kernel "
                       "bandwidth", color_discrete_sequence=PALETTE)
mx = float(max(S.raw_median.max(), S.cond_median.max()))
fig.add_shape(type="line", x0=0, y0=0, x1=mx, y1=mx, line=dict(dash="dash"))
fig.update_traces(textposition="top center")
fig.update_layout(height=520, xaxis_title="raw median err km",
                  yaxis_title="bandwidth-conditioned median err km")
fig.show()
print(f"spread of the sector effect shrinks by {S.shrinkage.iloc[0]:.1%} once "
      f"bandwidth is held fixed.")
print("Near 100%: sector is a proxy for footprint width and belongs in the "
      "PRIOR on bandwidth, not as a separate feature. Well below: sector "
      "carries something bandwidth misses and earns its own term.")

---
## 7. The locatability gate

The product. For a single-neighbour case, output a **calibrated probability
that the target is within R km**, not a point estimate. Fit on features known
without the answer — bandwidth, neighbour degree, hub flag, sector prior,
amount, tenure — and check calibration on a held-out split.

The number that decides whether this ships: **what fraction of degree-1 cases
can be flagged locatable at high precision, and how many are they?**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score, precision_recall_curve

GATE_R = 50
feat_sdf = (L.select(
        F.log1p(F.col("kc_bw_km")).alias("f_log_bw"),
        F.log1p(F.col("kc_S")).alias("f_log_ncp"),
        F.log1p(F.coalesce(F.col("b_deg"), F.lit(0))).alias("f_log_deg"),
        F.log1p(F.col("amt")).alias("f_log_amt"),
        F.col("n_months").cast("double").alias("f_months"),
        F.col("b_is_hub").cast("double").alias("f_hub"),
        F.coalesce(F.col("b_naics2"), F.lit("NA")).alias("naics2"),
        (F.col("err_cnt_km") <= GATE_R).cast("int").alias("y"),
        F.col("err_cnt_km"))
     .filter(F.col("f_log_bw").isNotNull()))
# a large but bounded pull; the gate is a 6-feature logit, not a deep model
frac = min(1.0, 3_000_000 / max(feat_sdf.count(), 1))
G = to_pd(feat_sdf.sample(False, frac, seed=SEED), "gate features")
G = G.dropna(subset=["f_log_bw", "y"])
print(f"gate sample: {len(G):,} | base rate within {GATE_R} km: {G.y.mean():.2%}")

# sector enters as a target-encoded PRIOR on locatability, fitted on train only
tr, te = train_test_split(G, test_size=0.3, random_state=SEED, stratify=G.y)
enc = tr.groupby("naics2")["y"].agg(["mean", "size"])
gm = tr.y.mean()
enc["sm"] = (enc["mean"]*enc["size"] + gm*200) / (enc["size"] + 200)  # shrunk
for d in (tr, te):
    d["f_naics_prior"] = d.naics2.map(enc["sm"]).fillna(gm)

FEATS = ["f_log_bw", "f_log_ncp", "f_log_deg", "f_log_amt", "f_months",
         "f_hub", "f_naics_prior"]
clf = LogisticRegression(max_iter=2000, C=1.0)
clf.fit(tr[FEATS].fillna(0), tr.y)
te["p"] = clf.predict_proba(te[FEATS].fillna(0))[:, 1]
auc = roc_auc_score(te.y, te.p)
print(f"\nAUC = {auc:.4f}")
print(pd.Series(clf.coef_[0], index=FEATS).sort_values()
        .rename("coefficient").to_string())

# bandwidth alone, for comparison — is the rest of the feature set earning?
clf1 = LogisticRegression(max_iter=2000).fit(tr[["f_log_bw"]], tr.y)
auc1 = roc_auc_score(te.y, clf1.predict_proba(te[["f_log_bw"]])[:, 1])
print(f"\nbandwidth-only AUC = {auc1:.4f}   (full model {auc:.4f})")
print("If the gap is small, ship the one-feature version: it is auditable, "
      "and explainability is a governance precondition here.")

In [ ]:
# Calibration and the operating point.
pt, pp = calibration_curve(te.y, te.p, n_bins=20, strategy="quantile")
fig = go.Figure()
fig.add_scatter(x=pp, y=pt, mode="lines+markers", name="model",
                line=dict(width=3, color=PALETTE[0]))
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash"))
fig.update_layout(height=460, title=f"Gate calibration — predicted vs actual "
                  f"P(within {GATE_R} km)", xaxis_title="predicted",
                  yaxis_title="observed")
fig.show()

prec, rec, thr = precision_recall_curve(te.y, te.p)
ops = []
for target_prec in (0.70, 0.80, 0.90, 0.95):
    ok = np.where(prec[:-1] >= target_prec)[0]
    if len(ok):
        i = ok[np.argmax(rec[ok])]
        cov = float((te.p >= thr[i]).mean())
        ops.append({"target_precision": target_prec, "threshold": thr[i],
                    "achieved_precision": prec[i], "recall": rec[i],
                    "share_of_pairs_flagged": cov})
OPS = pd.DataFrame(ops)
print(OPS.to_string(index=False))
print(f"\nRead 'share_of_pairs_flagged' as the size of the locatable "
      f"population. {GATE_R} km at high precision on a modest share is a real "
      f"asset; a point estimate for everyone is not.")
OPS.to_csv(os.path.join(OUT, "gate_operating_points.csv"), index=False)

---
## 8. Error vs evidence — how much does a second neighbour buy?

For targets with several located neighbours, draw a random permutation and
take running prefixes: the estimate after the first *k* neighbours, for every
*k* at once. One permutation therefore yields the whole curve, which is what
makes this affordable in Spark.

Three weighting schemes are accumulated in the same pass:

- **uniform** — every neighbour counts equally
- **amount** — Block F's convention, carried here to be falsified
- **inverse-variance** — `1/bandwidth²`, so a hub self-mutes

Plus `tightest-k`: ignore the random order and take the *k* narrowest kernels.
If that beats the mixture, evidence **selection** matters more than evidence
combination — which would change the product design.

In [ ]:
K = L.select("a", "b", "kc_lat", "kc_lon", "kc_bw_km", "amt",
             "a_lat", "a_lon").filter(F.col("kc_lat").isNotNull())
K = (unit_cols(K, "kc_lat", "kc_lon", pre="k")
       .withColumn("w_uni", F.lit(1.0))
       .withColumn("w_amt", F.col("amt"))
       .withColumn("w_ivr", F.lit(1.0) /
                   F.greatest(F.col("kc_bw_km"), F.lit(1.0))**2))

def prefix_curve(df, order_col, tag, reps):
    out = []
    for rep in range(reps):
        d = df.withColumn("_o", F.rand(SEED + rep) if order_col is None
                          else F.col(order_col))
        w = W.partitionBy("a").orderBy("_o")
        d = d.withColumn("k", F.row_number().over(w)).filter(F.col("k") <= K_MAX)
        wr = w.rowsBetween(W.unboundedPreceding, W.currentRow)
        for s in ("uni", "amt", "ivr"):
            for ax in ("x", "y", "z"):
                d = d.withColumn(f"{s}{ax}",
                                 F.sum(F.col(f"w_{s}")*F.col(f"ku{ax}")).over(wr))
        for s in ("uni", "amt", "ivr"):
            n = F.sqrt(F.col(f"{s}x")**2 + F.col(f"{s}y")**2 + F.col(f"{s}z")**2)
            d = (d.withColumn(f"{s}_lat", F.degrees(F.asin(F.col(f"{s}z")/n)))
                   .withColumn(f"{s}_lon", F.degrees(F.atan2(F.col(f"{s}y"),
                                                             F.col(f"{s}x"))))
                   .withColumn(f"e_{s}", gc_km(F.col("a_lat"), F.col("a_lon"),
                                               F.col(f"{s}_lat"),
                                               F.col(f"{s}_lon"))))
        g = (d.groupBy("k").agg(F.count("*").alias("n"),
                *[x for s in ("uni", "amt", "ivr") for x in
                  (F.expr(f"percentile_approx(e_{s},0.5)").alias(f"med_{s}"),
                   F.expr(f"percentile_approx(e_{s},0.9)").alias(f"p90_{s}"),
                   F.avg((F.col(f"e_{s}") <= 50).cast("double")).alias(f"hit50_{s}"))]))
        r = to_pd(g, f"{tag} rep{rep}"); r["rep"] = rep; r["order"] = tag
        out.append(r)
    return pd.concat(out, ignore_index=True)

CURVE = pd.concat([prefix_curve(K, None, "random", HARNESS_REPS),
                   prefix_curve(K, "kc_bw_km", "tightest_first", 1)],
                  ignore_index=True)
CURVE.to_csv(os.path.join(OUT, "error_vs_k.csv"), index=False)
CV = CURVE.groupby(["order", "k"]).mean(numeric_only=True).reset_index()
print(CV[CV.order == "random"][["k","n","med_uni","med_amt","med_ivr",
                                "hit50_ivr"]].to_string(index=False))

In [ ]:
long = CV.melt(id_vars=["order", "k"],
               value_vars=["med_uni", "med_amt", "med_ivr"],
               var_name="scheme", value_name="median_err_km")
fig = px.line(long, x="k", y="median_err_km", color="scheme",
              line_dash="order", markers=True,
              title="Error vs number of located neighbours — weighting schemes "
                    f"and selection order ({VERSION}, {WINDOW})",
              color_discrete_sequence=PALETTE)
fig.add_hline(y=BASELINE["median_km"], line_dash="dot", line_color="crimson",
              annotation_text="footprint prior")
fig.update_layout(height=470, xaxis_title="k located neighbours used",
                  yaxis_title="median error km")
fig.show()

fig = px.line(CV, x="k", y="hit50_ivr", color="order", markers=True,
              title="Share of targets located within 50 km, by evidence count",
              color_discrete_sequence=PALETTE)
fig.update_layout(height=400, yaxis_tickformat=".0%",
                  xaxis_title="k located neighbours used")
fig.show()
print("Where the random curve flattens is the point at which more evidence "
      "stops paying. Where tightest_first sits ABOVE random at low k is the "
      "value of selecting evidence rather than pooling it.")

---
## 9. The hub locating-power registry  *(run this at `VERSION = "V0"`)*

The roadmap treats hubs as a flat exclusion list. That is wrong for this
problem. A card processor has a national cloud and locates nothing. A county
tax office, a municipal utility, a regional grocery chain have clouds the size
of the geography they serve — a **state-sized kernel is weak but not useless**,
and at degree 1 weak evidence beats the national prior.

So the output here is not a list to drop. It is a **labelled registry with a
locating radius per hub**, which is the shape the Hub Node Registry was always
supposed to have.

In [ ]:
HUBP = (L.filter(F.col("b_is_hub") == 1)
          .groupBy("b", "b_name", "b_naics2", "b_node_type", "b_deg")
          .agg(F.count("*").alias("n_located_cp"),
               F.avg("kc_bw_km").alias("bw_km"),
               F.expr("percentile_approx(err_cnt_km,0.5)").alias("median_err_km"),
               F.avg((F.col("err_cnt_km") <= 250).cast("double")).alias("hit_250"),
               F.avg((F.col("err_cnt_km") <= 50).cast("double")).alias("hit_50"))
          .filter(F.col("n_located_cp") >= 50))
HP = to_pd(HUBP.orderBy("median_err_km"), "hub registry", max_rows=500_000)
HP["locating_class"] = pd.cut(HP.median_err_km, [-1, 25, 100, 400, 1e9],
                              labels=["PIN", "METRO", "REGION", "NONE"])
HP.to_csv(os.path.join(OUT, "hub_locating_registry.csv"), index=False)
print(HP.locating_class.value_counts().to_string())
print("\n--- most informative hubs ---")
print(HP.head(20)[["b_name","b_naics2","b_deg","n_located_cp","bw_km",
                   "median_err_km","hit_250"]].to_string(index=False))
print("\n--- least informative (these are the true exclusions) ---")
print(HP.tail(10)[["b_name","b_naics2","b_deg","bw_km","median_err_km"]]
        .to_string(index=False))

In [ ]:
fig = px.scatter(HP, x="b_deg", y="median_err_km", color="locating_class",
                 size="n_located_cp", hover_data=["b_name", "b_naics2", "bw_km"],
                 log_x=True, log_y=True, color_discrete_sequence=PALETTE,
                 title="Hub locating power — degree does NOT determine whether "
                       "a hub carries location")
for r in HIT_RADII_KM:
    fig.add_hline(y=r, line_dash="dot", line_color="grey")
fig.update_layout(height=520, xaxis_title="hub degree",
                  yaxis_title="median degree-1 error km")
fig.show()

sec = (HP.groupby("b_naics2")
         .agg(n=("b", "size"), median_err=("median_err_km", "median"))
         .query("n >= 5").sort_values("median_err"))
fig = px.bar(sec.reset_index(), x="b_naics2", y="median_err",
             hover_data=["n"], color_discrete_sequence=PALETTE,
             title="Hub sectors by locating power (government / utilities "
                   "should sit left if the hypothesis holds)")
fig.update_layout(height=400, yaxis_title="median error km")
fig.show()
print(sec.to_string())

---
## 10. Is the ground truth itself trustworthy?

Error is measured against the **registered pin**, and the previous notebook
showed the pin is not always where the business is. For a wide-footprint firm
the pin is noisy truth, so some of the measured error is the target's, not the
estimator's.

Re-scoring on the subset where `geo_registered_vs_flow_km` is small isolates
that. If accuracy improves sharply on the clean set, the headline numbers are
**pessimistic** — and the honest figure to quote is the clean one, with the
caveat that counterparties will resemble the full set more than the clean one.

In [ ]:
GAP = ATTR.select(F.col("node").alias("a"),
                  F.col("geo_registered_vs_flow_km").alias("a_gap"),
                  F.col("geo_spread_km").alias("a_spread"))
LG = L.join(GAP, "a", "left")
rows = []
for name, cond in [("all targets", F.lit(True)),
                   ("pin representative", F.col("a_gap") <= REPRESENTATIVE_GAP_KM),
                   ("pin unrepresentative", F.col("a_gap") > REPRESENTATIVE_GAP_KM),
                   ("target is local (<50km spread)", F.col("a_spread") < 50)]:
    r = (LG.filter(cond).agg(
            F.count("*").alias("n"),
            F.expr("percentile_approx(err_cnt_km,0.5)").alias("median_km"),
            F.expr("percentile_approx(err_cnt_km,0.9)").alias("p90_km"),
            *[F.avg((F.col("err_cnt_km") <= x).cast("double")).alias(f"hit_{x}")
              for x in HIT_RADII_KM]).first().asDict())
    rows.append({"truth_set": name, **r})
TRUTH = pd.DataFrame(rows)
print(TRUTH.to_string(index=False))
TRUTH.to_csv(os.path.join(OUT, "truth_sensitivity.csv"), index=False)

In [ ]:
# Out-of-footprint check: every observed neighbour is a PNC customer, so every
# prediction is pulled toward the PNC footprint. Targets registered outside the
# core states are the closest available analogue to a counterparty that banks
# elsewhere — which is most of them.
core = [r[0] for r in (ATTR.filter("has_geo = 1").groupBy("state").count()
                       .orderBy(F.desc("count")).limit(6).collect())]
print(f"core footprint states: {core}")
ST = (L.join(ATTR.select(F.col("node").alias("a"),
                         F.col("state").alias("a_state")), "a", "left")
        .withColumn("in_core", F.col("a_state").isin(core))
        .groupBy("in_core").agg(
            F.count("*").alias("n"),
            F.expr("percentile_approx(err_cnt_km,0.5)").alias("median_km"),
            F.avg((F.col("err_cnt_km") <= 50).cast("double")).alias("hit_50")))
OOF = to_pd(ST, "in/out of footprint")
print(OOF.to_string(index=False))
print("\nThe gap between these two rows is the size of the footprint bias — "
      "the amount by which customer-derived accuracy OVERSTATES what to expect "
      "on real counterparties. Quote it alongside every accuracy figure.")
OOF.to_csv(os.path.join(OUT, "footprint_bias.csv"), index=False)

In [ ]:
# Persist this configuration's headline numbers for the §11 comparison.
summary = {
    "version": VERSION, "window": WINDOW, "months": len(WIN_MONTHS),
    "pairs": int(d1["n"]), "hub_nodes": int(n_hub),
    "prior_median_km": float(BASELINE["median_km"]),
    "d1_median_cnt_km": float(d1["cnt_median"]),
    "d1_median_amt_km": float(d1["amt_median"]),
    **{f"d1_hit_{r}": float(d1[f"cnt_hit_{r}"]) for r in HIT_RADII_KM},
    "gate_auc": float(auc), "gate_auc_bw_only": float(auc1),
}
for k in (1, 3, 5, 10, 20):
    row = CV[(CV.order == "random") & (CV.k == k)]
    if len(row):
        summary[f"k{k}_median_ivr_km"] = float(row.med_ivr.iloc[0])
        summary[f"k{k}_hit50_ivr"] = float(row.hit50_ivr.iloc[0])
with open(os.path.join(OUT, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

---
## 11. Comparing configurations

Run the notebook four times — `V0`/`P99_9` × `last3`/`all` — then this section
reads the saved summaries. Two questions:

- **`last3` vs `all`.** Pooling 23 months adds edges and stales the address
  attribution. The difference is the **net**, not the staleness cost alone; if
  `all` wins, the extra evidence more than pays for the movers.
- **`V0` vs `P99_9`.** Whether hubs help *in aggregate*. Note this can be
  negative overall while §9 shows specific hubs are excellent — which is the
  argument for a graded registry over a flat exclusion.

In [ ]:
rows = []
for d in sorted(os.listdir(OUT_ROOT)):
    p = os.path.join(OUT_ROOT, d, "summary.json")
    if os.path.exists(p):
        rows.append(json.load(open(p)))
if not rows:
    print("no configurations written yet")
else:
    CMP = pd.DataFrame(rows).sort_values(["version", "window"])
    print(CMP.to_string(index=False))
    CMP.to_csv(os.path.join(OUT_ROOT, "config_comparison.csv"), index=False)
    if len(CMP) > 1:
        m = CMP.melt(id_vars=["version", "window"],
                     value_vars=[c for c in CMP.columns
                                 if c.startswith(("k", "d1_hit"))
                                 and "hit" in c],
                     var_name="metric", value_name="value")
        fig = px.bar(m, x="metric", y="value", color="window",
                     facet_row="version", barmode="group",
                     color_discrete_sequence=PALETTE,
                     title="Hit rates across configurations — window (staleness "
                           "vs volume) and rung (hubs in / out)")
        fig.update_layout(height=560, yaxis_tickformat=".0%")
        fig.show()

---
## 12. Reading the results

**The three numbers that decide the programme.**

1. `d1_hit_50` — the share of single-neighbour cases landing within 50 km with
   no gate at all. This is the raw regime.
2. `share_of_pairs_flagged` at 90% precision (§7) — the share that can be
   located *and known to be located*. **This is the deliverable size.** If it
   is 15%, then 15% of counterparties get a CBSA and 85% get an honest
   `unknown`, which is a usable asset. A point estimate for 100% is not.
3. The in-core vs out-of-core gap (§10) — how much the customer-derived numbers
   overstate what counterparties will do.

**What each section settles.**

| Result | Consequence |
|---|---|
| Bandwidth-only AUC ≈ full AUC | ship the one-feature gate; it is auditable line by line, which matters because this feeds prospecting |
| Sector effect vanishes conditional on bandwidth | NAICS belongs in the **prior on kernel width**, not as its own term — and the sparse-reach neighbours can be shrunk toward it |
| `med_amt` > `med_uni` | dollars do not carry location; Block F's amount weighting is right for footprints and wrong here |
| `tightest_first` beats `random` at low k | **selection beats combination** — the product should choose which neighbour to trust, not average them |
| curve flat beyond k≈3–5 | the marginal located neighbour stops paying early; effort should go to the gate, not to acquiring more edges |
| hubs split into PIN/METRO/REGION/NONE | replace the flat exclusion list with the graded registry |

**What this cannot tell you.**

- **The distribution shift is real and one-directional.** Counterparties bank
  elsewhere, which correlates with sitting outside the PNC footprint, and their
  observed neighbour sets are a more biased sample of their true counterparties.
  Every number here is **optimistic**. The out-of-footprint split bounds it; it
  does not remove it.
- **`scope = on_us_c2c`.** A counterparty's real counterparty cloud is mostly
  invisible. We are inferring position from the PNC-visible slice only.
- **No external ground truth yet.** The FI pinning registry (FDIC Summary of
  Deposits, NCUA) is the one available set of counterparties with known
  locations. It is small and skewed to financial institutions, but it is drawn
  from the *right population* — treat it as a precondition for briefing any
  accuracy figure outside the team.

**Sequencing.**

1. Run all four configurations; settle window and rung before tuning anything.
2. Freeze the gate at a precision target, and emit `(cell, radius, p)` — never
   a bare lat/lon.
3. Validate against the FI registry; report the customer-to-counterparty gap.
4. Only then consider learned models. The estimator here is one round of
   message passing with hand-set weights, which makes it the ablation baseline
   a GNN has to beat — and a compliance read on inferring locations for
   non-customer prospecting is a precondition either way.